# Cap10 Three-Fold BC Slice Screen

This notebook tests whether the qlib-aligned SOXLBacktest buysell target contract can be learned alongside the existing cap10 fold-local joint target contract. It is a BC learnability gate only: no actor is retained, no promotion is allowed, and any later checkpoint/PPO work must still pass walk-forward replay gates.

In [1]:
from __future__ import annotations

import json
import math
import os
import random
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    raise RuntimeError(f'Could not locate repo root from {Path.cwd()}')
os.chdir(ROOT)

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.config import load_profile

EXPERIMENT = 'cap10_three_fold_bc_slice_screen_v1'
OUT_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
JOINT_SOURCE_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_foldlocal_joint_target_contract_v1'
WF2020_SOURCE_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_wf2020_soxlbacktest_target_contract_v1'
JOINT_CONTRACT_PATH = JOINT_SOURCE_DIR / 'joint_target_contract.csv'
WF2020_CONTRACT_PATH = WF2020_SOURCE_DIR / 'soxlbacktest_wf2020_target_contract.csv'
JOINT_SELECTION_PATH = JOINT_SOURCE_DIR / 'selection.json'
WF2020_SELECTION_PATH = WF2020_SOURCE_DIR / 'selection.json'
WF2020_SUMMARY_PATH = WF2020_SOURCE_DIR / 'summary.json'
FRAME_PATH = ROOT / 'data' / 'soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'risk_budget_usage_v1'

OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 15.0
GLOBAL_MIN_ACCURACY = 0.70
FOLD_MIN_ACCURACY = 0.70
SLICE_MIN_ACCURACY = 0.60
REQUIRED_FOLDS = ('wf_2020_train', 'wf_2022_train', 'wf_2023_2024_train')
REQUIRED_SLICES = {
    'release_gate_state': 20,
    'is_downtrend_state': 100,
    'drawdown_control_state': 20,
}
SLICE_COLS = tuple(REQUIRED_SLICES) + ('is_rebound_cluster',)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n')

print('root', ROOT)
print('out_dir', OUT_DIR)

root /Users/rziiiii/dev/workspace
out_dir /Users/rziiiii/dev/workspace/runtime/model_selection/card/cap10_three_fold_bc_slice_screen_v1


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
def _load_selection_or_summary(selection_path: Path, summary_path: Path | None = None) -> dict[str, Any]:
    if selection_path.exists():
        return json.loads(selection_path.read_text())
    if summary_path is not None and summary_path.exists():
        return json.loads(summary_path.read_text())
    return {}


def _prepare_contract(path: Path, source_contract_id: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    for col in ('is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'drawdown_control_state'):
        if col in df.columns:
            df[col] = df[col].fillna(False).astype(bool)
        else:
            df[col] = False
    df['source_contract_id'] = source_contract_id
    return df

joint_contract = _prepare_contract(JOINT_CONTRACT_PATH, 'cap10_foldlocal_joint_target_contract_v1')
wf2020_contract = _prepare_contract(WF2020_CONTRACT_PATH, 'cap10_wf2020_soxlbacktest_target_contract_v1')
contract = pd.concat([wf2020_contract, joint_contract], ignore_index=True, sort=False)
fold_order = {fold: idx for idx, fold in enumerate(REQUIRED_FOLDS)}
contract['_fold_order'] = contract['fold'].map(fold_order).fillna(999).astype(int)
contract = contract.sort_values(['_fold_order', 'datetime', 'source_contract_id']).drop(columns=['_fold_order']).reset_index(drop=True)
contract.to_csv(OUT_DIR / 'combined_target_contract.csv', index=False)

joint_selection = _load_selection_or_summary(JOINT_SELECTION_PATH)
wf2020_selection = _load_selection_or_summary(WF2020_SELECTION_PATH, WF2020_SUMMARY_PATH)
frame = pd.read_pickle(FRAME_PATH)
profile = load_profile(PROFILE_NAME)

folds = sorted(contract['fold'].dropna().astype(str).unique().tolist(), key=lambda item: fold_order.get(item, 999))
missing_required_folds = [fold for fold in REQUIRED_FOLDS if fold not in folds]
extra_folds = [fold for fold in folds if fold not in REQUIRED_FOLDS]
unique_dates = int(contract['datetime'].nunique())
contract_rows = int(len(contract))
pooled_dataset = build_state_objective_bc_dataset_from_frame(frame, targets=contract, profile=profile)

fold_dataset_rows = []
for fold in folds:
    fold_targets = contract[contract['fold'].astype(str).eq(fold)].copy()
    dataset = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile)
    metadata = dataset.metadata
    fold_dataset_rows.append({
        'fold': fold,
        'source_contract_id': ','.join(sorted(fold_targets['source_contract_id'].dropna().astype(str).unique())),
        'contract_rows': int(len(fold_targets)),
        'dataset_samples': int(len(dataset.actions)),
        'dropped_rows_before_dataset': int(len(fold_targets) - len(dataset.actions)),
        'start': pd.Timestamp(fold_targets['datetime'].min()).date().isoformat(),
        'end': pd.Timestamp(fold_targets['datetime'].max()).date().isoformat(),
        'release_gate_samples': int(metadata.get('release_gate_state', pd.Series(False, index=metadata.index)).fillna(False).astype(bool).sum()),
        'downtrend_samples': int(metadata.get('is_downtrend_state', pd.Series(False, index=metadata.index)).fillna(False).astype(bool).sum()),
        'drawdown_control_samples': int(metadata.get('drawdown_control_state', pd.Series(False, index=metadata.index)).fillna(False).astype(bool).sum()),
        'rebound_cluster_samples': int(metadata.get('is_rebound_cluster', pd.Series(False, index=metadata.index)).fillna(False).astype(bool).sum()),
    })

required_fold_coverage = not missing_required_folds
wf_2020_coverage = 'wf_2020_train' in folds and not any(row['dataset_samples'] <= 0 for row in fold_dataset_rows if row['fold'] == 'wf_2020_train')

dataset_summary = {
    'experiment': EXPERIMENT,
    'profile': PROFILE_NAME,
    'feature_frame': str(FRAME_PATH.relative_to(ROOT)),
    'source_contracts': [
        str(WF2020_CONTRACT_PATH.relative_to(ROOT)),
        str(JOINT_CONTRACT_PATH.relative_to(ROOT)),
    ],
    'combined_contract': 'combined_target_contract.csv',
    'contract_rows': contract_rows,
    'unique_dates': unique_dates,
    'pooled_bc_dataset_samples_after_date_dedup': int(len(pooled_dataset.actions)),
    'date_dedup_gap_rows': int(contract_rows - len(pooled_dataset.actions)),
    'folds': folds,
    'required_folds': list(REQUIRED_FOLDS),
    'required_fold_coverage': required_fold_coverage,
    'missing_required_folds': missing_required_folds,
    'extra_folds': extra_folds,
    'wf_2020_coverage': bool(wf_2020_coverage),
    'fold_dataset_rows': fold_dataset_rows,
    'source_decisions': {
        'cap10_foldlocal_joint_target_contract_v1': joint_selection.get('decision'),
        'cap10_wf2020_soxlbacktest_target_contract_v1': wf2020_selection.get('decision'),
    },
    'source_next_allowed': {
        'cap10_foldlocal_joint_target_contract_v1': joint_selection.get('bc_screen_allowed_next'),
        'cap10_wf2020_soxlbacktest_target_contract_v1': wf2020_selection.get('bc_screen_allowed_next'),
    },
}
write_json(OUT_DIR / 'dataset_summary.json', dataset_summary)
pd.DataFrame(fold_dataset_rows).to_csv(OUT_DIR / 'fold_dataset_audit.csv', index=False)

display(pd.DataFrame(fold_dataset_rows))
print(json.dumps(json_safe(dataset_summary), indent=2, allow_nan=False))

,fold,source_contract_id,contract_rows,dataset_samples,dropped_rows_before_dataset,start,end,release_gate_samples,downtrend_samples,drawdown_control_samples,rebound_cluster_samples
0,wf_2020_train,cap10_wf2020_soxlbacktest_target_contract_v1,2470,2157,313,2010-03-11,2019-12-31,36,640,640,36
1,wf_2022_train,cap10_foldlocal_joint_target_contract_v1,505,505,0,2020-01-02,2021-12-31,40,234,49,66
2,wf_2023_2024_train,cap10_foldlocal_joint_target_contract_v1,756,755,1,2020-01-02,2022-12-30,40,477,49,66


{
  "experiment": "cap10_three_fold_bc_slice_screen_v1",
  "profile": "risk_budget_usage_v1",
  "feature_frame": "data/soxl_profile_balanced_rule_residual_v22.pkl",
  "source_contracts": [
    "runtime/model_selection/card/cap10_wf2020_soxlbacktest_target_contract_v1/soxlbacktest_wf2020_target_contract.csv",
    "runtime/model_selection/card/cap10_foldlocal_joint_target_contract_v1/joint_target_contract.csv"
  ],
  "combined_contract": "combined_target_contract.csv",
  "contract_rows": 3731,
  "unique_dates": 3226,
  "pooled_bc_dataset_samples_after_date_dedup": 2912,
  "date_dedup_gap_rows": 819,
  "folds": [
    "wf_2020_train",
    "wf_2022_train",
    "wf_2023_2024_train"
  ],
  "required_folds": [
    "wf_2020_train",
    "wf_2022_train",
    "wf_2023_2024_train"
  ],
  "required_fold_coverage": true,
  "missing_required_folds": [],
  "extra_folds": [],
  "wf_2020_coverage": true,
  "fold_dataset_rows": [
    {
      "fold": "wf_2020_train",
      "source_contract_id": "cap10_wf20

## BC Weight Screen

The screen evaluates three static weighting policies. Each policy trains a fresh fold-local BC actor for each required training fold and then checks global fit, per-fold fit, and sparse-state slice fit.

In [3]:
TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)
LONG_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=600, lr=0.001, batch_size=128, seed=SEED)
VERY_LONG_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=1000, lr=0.001, batch_size=128, seed=SEED)
WEIGHT_CONFIGS: list[dict[str, Any]] = [
    {
        'config_name': 'plain',
        'complexity': 0,
        'sample_weight_config': None,
        'description': 'No static sample weighting; tests whether the raw three-fold target contract is already learnable.',
    },
    {
        'config_name': 'release4_downtrend2',
        'complexity': 1,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=4.0,
            downtrend_multiplier=2.0,
            weight_cap=20.0,
        ),
        'description': 'Moderate sparse-state weighting for rebound/release plus downtrend control.',
    },
    {
        'config_name': 'release8_rebound4_downtrend2',
        'complexity': 2,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=4.0,
            downtrend_multiplier=2.0,
            release_gate_multiplier=8.0,
            weight_cap=25.0,
        ),
        'description': 'Adds explicit release-gate weighting while keeping moderate rebound/downtrend weights.',
    },
    {
        'config_name': 'release12_rebound4_downtrend2',
        'complexity': 3,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=4.0,
            downtrend_multiplier=2.0,
            release_gate_multiplier=12.0,
            weight_cap=30.0,
        ),
        'description': 'Stronger explicit release-gate weighting to test the wf_2020 sparse-state failure.',
    },
    {
        'config_name': 'release10_rebound4_downtrend2_steps600',
        'complexity': 4,
        'train_config': LONG_TRAIN_CONFIG,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=4.0,
            downtrend_multiplier=2.0,
            release_gate_multiplier=10.0,
            weight_cap=30.0,
        ),
        'description': 'Moderate release-gate weighting with a longer BC schedule; schedule sanity check showed this can preserve fold fit.',
    },
    {
        'config_name': 'release12_rebound4_downtrend2_steps600',
        'complexity': 5,
        'train_config': LONG_TRAIN_CONFIG,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=4.0,
            downtrend_multiplier=2.0,
            release_gate_multiplier=12.0,
            weight_cap=30.0,
        ),
        'description': 'Stronger release-gate weighting with the longer BC schedule.',
    },
    {
        'config_name': 'release9_rebound4_downtrend2_steps1000',
        'complexity': 6,
        'train_config': VERY_LONG_TRAIN_CONFIG,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=4.0,
            downtrend_multiplier=2.0,
            release_gate_multiplier=9.0,
            weight_cap=30.0,
        ),
        'description': 'Very-long schedule guardrail to test whether remaining failure was only optimization time.',
    },
    {
        'config_name': 'release16_rebound6_downtrend2_cb99',
        'complexity': 4,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=6.0,
            downtrend_multiplier=2.0,
            release_gate_multiplier=16.0,
            objective_weight_power=0.20,
            class_balanced_beta=0.99,
            weight_cap=35.0,
        ),
        'description': 'Release-heavy, class-balanced weighting for the tail action fit stress test.',
    },
    {
        'config_name': 'joint8_downtrend2_cb99',
        'complexity': 2,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=8.0,
            downtrend_multiplier=2.0,
            objective_weight_power=0.25,
            class_balanced_beta=0.99,
            weight_cap=20.0,
        ),
        'description': 'Stronger sparse-state and class-balanced weighting for tail action fit.',
    },
]

config_rows = []
for item in WEIGHT_CONFIGS:
    cfg = item['sample_weight_config']
    row = {k: v for k, v in item.items() if k not in {'sample_weight_config', 'train_config'}}
    row['sample_weight_config'] = None if cfg is None else asdict(cfg)
    row['train_config'] = asdict(item.get('train_config', TRAIN_CONFIG))
    config_rows.append(row)

pd.DataFrame(config_rows).to_csv(OUT_DIR / 'bc_weight_configs.csv', index=False)
display(pd.DataFrame(config_rows))

,config_name,complexity,description,sample_weight_config,train_config
0,plain,0,No static sample weighting; tests whether the ...,None,"{'steps': 300, 'lr': 0.001, 'batch_size': 128,..."
1,release4_downtrend2,1,Moderate sparse-state weighting for rebound/re...,"{'rebound_multiplier': 4.0, 'downtrend_multipl...","{'steps': 300, 'lr': 0.001, 'batch_size': 128,..."
2,release8_rebound4_downtrend2,2,Adds explicit release-gate weighting while kee...,"{'rebound_multiplier': 4.0, 'downtrend_multipl...","{'steps': 300, 'lr': 0.001, 'batch_size': 128,..."
3,release12_rebound4_downtrend2,3,Stronger explicit release-gate weighting to te...,"{'rebound_multiplier': 4.0, 'downtrend_multipl...","{'steps': 300, 'lr': 0.001, 'batch_size': 128,..."
4,release10_rebound4_downtrend2_steps600,4,Moderate release-gate weighting with a longer ...,"{'rebound_multiplier': 4.0, 'downtrend_multipl...","{'steps': 600, 'lr': 0.001, 'batch_size': 128,..."
5,release12_rebound4_downtrend2_steps600,5,Stronger release-gate weighting with the longe...,"{'rebound_multiplier': 4.0, 'downtrend_multipl...","{'steps': 600, 'lr': 0.001, 'batch_size': 128,..."
6,release9_rebound4_downtrend2_steps1000,6,Very-long schedule guardrail to test whether r...,"{'rebound_multiplier': 4.0, 'downtrend_multipl...","{'steps': 1000, 'lr': 0.001, 'batch_size': 128..."
7,release16_rebound6_downtrend2_cb99,4,"Release-heavy, class-balanced weighting for th...","{'rebound_multiplier': 6.0, 'downtrend_multipl...","{'steps': 300, 'lr': 0.001, 'batch_size': 128,..."
8,joint8_downtrend2_cb99,2,Stronger sparse-state and class-balanced weigh...,"{'rebound_multiplier': 8.0, 'downtrend_multipl...","{'steps': 300, 'lr': 0.001, 'batch_size': 128,..."


In [4]:
def _metric_loss(value: float, threshold: float) -> float:
    if value is None or not math.isfinite(float(value)):
        return 1.0
    return max(0.0, float(threshold) - float(value))


def _sample_loss(samples: int, min_samples: int) -> float:
    return 0.0 if int(samples) >= int(min_samples) else (int(min_samples) - int(samples)) / max(int(min_samples), 1)

screen_rows: list[dict[str, Any]] = []
slice_rows: list[dict[str, Any]] = []
fold_rows: list[dict[str, Any]] = []
action_rows: list[dict[str, Any]] = []

for item in WEIGHT_CONFIGS:
    config_name = item['config_name']
    complexity = int(item['complexity'])
    sample_weight_config = item['sample_weight_config']
    train_config = item.get('train_config', TRAIN_CONFIG)
    per_fold_accuracy: list[float] = []
    per_fold_samples: list[int] = []
    per_slice_values: dict[str, list[dict[str, float | int]]] = {name: [] for name in REQUIRED_SLICES}
    config_loss = 0.0

    for fold in REQUIRED_FOLDS:
        fold_targets = contract[contract['fold'].astype(str).eq(fold)].copy()
        if fold_targets.empty:
            fold_rows.append({
                'config_name': config_name,
                'complexity': complexity,
                'fold': fold,
                'samples': 0,
                'loss': math.nan,
                'accuracy': math.nan,
                'fold_gate': False,
                'min_accuracy': FOLD_MIN_ACCURACY,
                'before_accuracy': math.nan,
                'after_accuracy': math.nan,
                'after_loss': math.nan,
            })
            config_loss += 1.0
            continue

        torch.manual_seed(SEED)
        np.random.seed(SEED)
        random.seed(SEED)
        dataset = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile)
        fit = fit_state_objective_bc_actor_from_frame(
            frame,
            targets=fold_targets,
            profile=profile,
            train_config=train_config,
            sample_weight_config=sample_weight_config,
        )
        fold_eval = evaluate_bc_actor(fit.actor, dataset.obs, dataset.actions)
        fold_accuracy = float(fold_eval['accuracy'])
        fold_samples = int(fold_eval['samples'])
        per_fold_accuracy.append(fold_accuracy)
        per_fold_samples.append(fold_samples)
        fold_pass = fold_accuracy >= FOLD_MIN_ACCURACY
        fold_rows.append({
            'config_name': config_name,
            'complexity': complexity,
            'fold': fold,
            'samples': fold_samples,
            'loss': float(fold_eval['loss']),
            'accuracy': fold_accuracy,
            'fold_gate': bool(fold_pass),
            'min_accuracy': FOLD_MIN_ACCURACY,
            'before_accuracy': float(fit.metrics['before_accuracy']),
            'after_accuracy': float(fit.metrics['after_accuracy']),
            'after_loss': float(fit.metrics['after_loss']),
        })
        config_loss += _metric_loss(fold_accuracy, FOLD_MIN_ACCURACY)

        metrics = evaluate_bc_actor_slice_fit(
            fit.actor,
            dataset.obs,
            dataset.actions,
            dataset.metadata,
            slice_cols=SLICE_COLS,
        )
        for row in metrics.to_dict('records'):
            row = dict(row)
            row.update({'config_name': config_name, 'complexity': complexity, 'fold': fold})
            min_samples = REQUIRED_SLICES.get(str(row['slice']), 0)
            row['min_samples'] = int(min_samples)
            row['min_accuracy'] = SLICE_MIN_ACCURACY if str(row['slice']) in REQUIRED_SLICES else None
            row['slice_gate'] = (
                str(row['slice']) not in REQUIRED_SLICES
                or (int(row['samples']) >= int(min_samples) and float(row['accuracy']) >= SLICE_MIN_ACCURACY)
            )
            slice_rows.append(row)
            if str(row['slice']) in REQUIRED_SLICES:
                per_slice_values[str(row['slice'])].append({
                    'samples': int(row['samples']),
                    'accuracy': float(row['accuracy']),
                })
                config_loss += _metric_loss(float(row['accuracy']), SLICE_MIN_ACCURACY)
                config_loss += _sample_loss(int(row['samples']), int(min_samples))

        dist = fit.action_distribution.copy()
        dist['config_name'] = config_name
        dist['complexity'] = complexity
        dist['fold'] = fold
        action_rows.extend(dist.to_dict('records'))

    weighted_accuracy = float(np.average(per_fold_accuracy, weights=per_fold_samples)) if per_fold_accuracy and sum(per_fold_samples) > 0 else math.nan
    min_fold_accuracy = float(min(per_fold_accuracy)) if per_fold_accuracy else math.nan
    aggregate: dict[str, Any] = {
        'config_name': config_name,
        'complexity': complexity,
        'description': item['description'],
        'folds': len(per_fold_accuracy),
        'required_folds': len(REQUIRED_FOLDS),
        'required_fold_coverage': bool(required_fold_coverage and len(per_fold_accuracy) == len(REQUIRED_FOLDS)),
        'missing_required_folds': ','.join(missing_required_folds),
        'wf_2020_coverage': bool(wf_2020_coverage),
        'total_samples': int(sum(per_fold_samples)),
        'weighted_accuracy': weighted_accuracy,
        'min_fold_accuracy': min_fold_accuracy,
        'global_accuracy_gate': bool(math.isfinite(weighted_accuracy) and weighted_accuracy >= GLOBAL_MIN_ACCURACY),
        'fold_accuracy_gate': bool(math.isfinite(min_fold_accuracy) and min_fold_accuracy >= FOLD_MIN_ACCURACY),
        'train_steps': int(train_config.steps),
        'lr': float(train_config.lr),
        'batch_size': int(train_config.batch_size),
    }
    config_loss += 1.0 if not aggregate['required_fold_coverage'] else 0.0
    slice_gate = True
    for slice_name, min_samples in REQUIRED_SLICES.items():
        values = per_slice_values[slice_name]
        min_acc = min((float(v['accuracy']) for v in values), default=float('nan'))
        min_seen = min((int(v['samples']) for v in values), default=0)
        aggregate[f'{slice_name}_min_accuracy'] = min_acc
        aggregate[f'{slice_name}_min_samples'] = int(min_seen)
        aggregate[f'{slice_name}_gate'] = bool(min_seen >= int(min_samples) and math.isfinite(min_acc) and min_acc >= SLICE_MIN_ACCURACY)
        slice_gate = slice_gate and bool(aggregate[f'{slice_name}_gate'])
    aggregate['slice_gate'] = bool(slice_gate)
    aggregate['bc_screen_gate'] = bool(
        aggregate['required_fold_coverage']
        and aggregate['global_accuracy_gate']
        and aggregate['fold_accuracy_gate']
        and slice_gate
    )
    aggregate['gate_loss'] = float(config_loss + complexity * 0.001)
    aggregate['ppo_screen_allowed_next'] = bool(aggregate['bc_screen_gate'])
    aggregate['training_allowed_from_this_card'] = False
    aggregate['promotion_allowed'] = False
    screen_rows.append(aggregate)

screen = pd.DataFrame(screen_rows).sort_values(['bc_screen_gate', 'gate_loss', 'complexity'], ascending=[False, True, True]).reset_index(drop=True)
slice_metrics = pd.DataFrame(slice_rows).sort_values(['config_name', 'fold', 'slice']).reset_index(drop=True)
fold_metrics = pd.DataFrame(fold_rows).sort_values(['config_name', 'fold']).reset_index(drop=True)
action_distribution = pd.DataFrame(action_rows).sort_values(['config_name', 'fold', 'action']).reset_index(drop=True)

screen.to_csv(OUT_DIR / 'bc_weight_screen.csv', index=False)
slice_metrics.to_csv(OUT_DIR / 'state_slice_bc_metrics.csv', index=False)
fold_metrics.to_csv(OUT_DIR / 'fold_bc_metrics.csv', index=False)
action_distribution.to_csv(OUT_DIR / 'bc_action_distribution.csv', index=False)

display(screen)
display(slice_metrics[slice_metrics['slice'].isin(['release_gate_state', 'is_downtrend_state', 'drawdown_control_state'])])

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


...

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


,config_name,complexity,description,folds,required_folds,required_fold_coverage,missing_required_folds,wf_2020_coverage,total_samples,weighted_accuracy,...,is_downtrend_state_gate,drawdown_control_state_min_accuracy,drawdown_control_state_min_samples,drawdown_control_state_gate,slice_gate,bc_screen_gate,gate_loss,ppo_screen_allowed_next,training_allowed_from_this_card,promotion_allowed
0,release10_rebound4_downtrend2_steps600,4,Moderate release-gate weighting with a longer ...,3,3,True,,True,3417,0.759145,...,True,0.870313,49,True,True,True,0.004000,True,False,False
1,release12_rebound4_downtrend2_steps600,5,Stronger release-gate weighting with the longe...,3,3,True,,True,3417,0.741879,...,True,0.846875,49,True,True,True,0.005000,True,False,False
2,release9_rebound4_downtrend2_steps1000,6,Very-long schedule guardrail to test whether r...,3,3,True,,True,3417,0.794849,...,True,0.885938,49,True,True,True,0.006000,True,False,False
3,release12_rebound4_downtrend2,3,Stronger explicit release-gate weighting to te...,3,3,True,,True,3417,0.684811,...,True,0.857812,49,True,True,False,0.051169,False,False,False
4,release8_rebound4_downtrend2,2,Adds explicit release-gate weighting while kee...,3,3,True,,True,3417,0.695054,...,True,0.854688,49,True,False,False,0.078851,False,False,False
5,release16_rebound6_downtrend2_cb99,4,"Release-heavy, class-balanced weighting for th...",3,3,True,,True,3417,0.649107,...,True,0.854688,49,True,True,False,0.104159,False,False,False
6,plain,0,No static sample weighting; tests whether the ...,3,3,True,,True,3417,0.745391,...,True,0.820312,49,True,False,False,0.127778,False,False,False
7,joint8_downtrend2_cb99,2,Stronger sparse-state and class-balanced weigh...,3,3,True,,True,3417,0.666959,...,True,0.877551,49,True,False,False,0.183085,False,False,False
8,release4_downtrend2,1,Moderate sparse-state weighting for rebound/re...,3,3,True,,True,3417,0.672520,...,True,0.734694,49,True,False,False,0.196616,False,False,False


,slice,samples,accuracy,loss,label_unique_actions,predicted_unique_actions,label_dominant_action,label_dominant_share,predicted_dominant_action,predicted_dominant_share,config_name,complexity,fold,min_samples,min_accuracy,slice_gate
1,drawdown_control_state,640,0.878125,1.028743,4,3,2,0.596875,2,0.643750,joint8_downtrend2_cb99,2,wf_2020_train,20,0.6,True
2,is_downtrend_state,640,0.878125,1.028743,4,3,2,0.596875,2,0.643750,joint8_downtrend2_cb99,2,wf_2020_train,100,0.6,True
4,release_gate_state,36,0.500000,1.428807,4,3,4,0.722222,4,0.527778,joint8_downtrend2_cb99,2,wf_2020_train,20,0.6,False
6,drawdown_control_state,49,0.979592,0.930256,3,3,4,0.857143,4,0.836735,joint8_downtrend2_cb99,2,wf_2022_train,20,0.6,True
7,is_downtrend_state,234,0.863248,1.038628,4,5,1,0.431624,1,0.401709,joint8_downtrend2_cb99,2,wf_2022_train,100,0.6,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,is_downtrend_state,234,0.837607,1.064366,4,4,1,0.431624,1,0.491453,release9_rebound4_downtrend2_steps1000,6,wf_2022_train,100,0.6,True
129,release_gate_state,40,1.000000,0.904832,1,1,4,1.000000,4,1.000000,release9_rebound4_downtrend2_steps1000,6,wf_2022_train,20,0.6,True
131,drawdown_control_state,49,0.979592,0.925247,3,3,4,0.857143,4,0.877551,release9_rebound4_downtrend2_steps1000,6,wf_2023_2024_train,20,0.6,True
132,is_downtrend_state,477,0.878407,1.027335,5,4,1,0.383648,1,0.417191,release9_rebound4_downtrend2_steps1000,6,wf_2023_2024_train,100,0.6,True


In [5]:
passed = screen[screen['bc_screen_gate'].astype(bool)]
if not passed.empty:
    selected = passed.sort_values(['complexity', 'gate_loss']).iloc[0].to_dict()
    decision = 'three_fold_bc_slice_ready_for_checkpoint_screen_no_promotion'
    next_mechanism = 'run_walkforward_checkpoint_or_ppo_screen_with_bc_warmstart_against_soxlbacktest_reference_and_drawdown_drift_stop_rules'
else:
    selected = screen.sort_values(['gate_loss', 'complexity']).iloc[0].to_dict()
    decision = 'three_fold_bc_slice_failed_no_ppo'
    next_mechanism = 'retune_target_contract_or_state_weighting_before_any_checkpoint_training'

selected_config = str(selected['config_name'])
selected_slices = slice_metrics[slice_metrics['config_name'].eq(selected_config)].copy()
selected_folds = fold_metrics[fold_metrics['config_name'].eq(selected_config)].copy()

failed_checks = []
if not bool(selected['required_fold_coverage']):
    failed_checks.append('required_fold_coverage')
if not bool(selected['global_accuracy_gate']):
    failed_checks.append('global_accuracy')
if not bool(selected['fold_accuracy_gate']):
    failed_checks.append('fold_accuracy')
for name in REQUIRED_SLICES:
    if not bool(selected[f'{name}_gate']):
        failed_checks.append(name)

slice_summary = {
    'experiment': EXPERIMENT,
    'selected_config': selected_config,
    'decision': decision,
    'bc_screen_gate': bool(selected['bc_screen_gate']),
    'failed_checks': failed_checks,
    'required_folds': list(REQUIRED_FOLDS),
    'missing_required_folds': missing_required_folds,
    'required_slices': REQUIRED_SLICES,
    'slice_min_accuracy': SLICE_MIN_ACCURACY,
    'global_min_accuracy': GLOBAL_MIN_ACCURACY,
    'fold_min_accuracy': FOLD_MIN_ACCURACY,
    'selected': selected,
    'selected_slice_rows': selected_slices.to_dict('records'),
    'selected_fold_rows': selected_folds.to_dict('records'),
}
write_json(OUT_DIR / 'bc_slice_summary.json', slice_summary)

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'selected_config': selected_config,
    'selected_gate_loss': float(selected['gate_loss']),
    'selected_weighted_accuracy': float(selected['weighted_accuracy']),
    'selected_min_fold_accuracy': float(selected['min_fold_accuracy']),
    'bc_screen_gate': bool(selected['bc_screen_gate']),
    'ppo_screen_allowed_next': bool(selected['bc_screen_gate']),
    'checkpoint_screen_allowed_next': bool(selected['bc_screen_gate']),
    'training_allowed_from_this_card': False,
    'actor_retention_allowed': False,
    'promotion_allowed': False,
    'promotion_blocker': 'BC slice fit is not a backtest; promotion still requires fixed-split and wf_2020/wf_2022/wf_2023_2024 replay with excess >= +30pp and MaxDD <= 15%.',
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'failed_checks': failed_checks,
    'next_mechanism': next_mechanism,
    'source_contracts': [
        str(WF2020_CONTRACT_PATH.relative_to(ROOT)),
        str(JOINT_CONTRACT_PATH.relative_to(ROOT)),
    ],
    'source_decisions': dataset_summary['source_decisions'],
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'cap10_wf2020_soxlbacktest_target_contract_v1 + cap10_foldlocal_joint_target_contract_v1',
    'reference_baseline': 'soxlbacktest_qlib_buysell_reference',
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'training_objective'],
    'external_sources': ['legacy_soxl_backtest_buysell_path', 'qlib_aligned_daily_panel'],
    'required_folds': list(REQUIRED_FOLDS),
    'missing_required_folds': missing_required_folds,
    'wf_2020_coverage': bool(wf_2020_coverage),
    'actor_retention': 'no actor retained; rerun this notebook to regenerate fold-local BC actors',
    'sidecars': [
        'combined_target_contract.csv',
        'dataset_summary.json',
        'fold_dataset_audit.csv',
        'bc_weight_configs.csv',
        'bc_weight_screen.csv',
        'state_slice_bc_metrics.csv',
        'fold_bc_metrics.csv',
        'bc_action_distribution.csv',
        'bc_slice_summary.json',
        'selection.json',
        'summary.json',
        'conclusion.md',
    ],
}
summary = {
    **selection,
    'screen_rows': screen.to_dict('records'),
}
write_json(OUT_DIR / 'selection.json', selection)
write_json(OUT_DIR / 'summary.json', summary)

print(json.dumps(json_safe(selection), indent=2, allow_nan=False))

{
  "experiment": "cap10_three_fold_bc_slice_screen_v1",
  "created_at": "2026-06-13T11:49:56.541279+00:00",
  "decision": "three_fold_bc_slice_ready_for_checkpoint_screen_no_promotion",
  "selected_config": "release10_rebound4_downtrend2_steps600",
  "selected_gate_loss": 0.004,
  "selected_weighted_accuracy": 0.7591454492244659,
  "selected_min_fold_accuracy": 0.7273991655076495,
  "bc_screen_gate": true,
  "ppo_screen_allowed_next": true,
  "checkpoint_screen_allowed_next": true,
  "training_allowed_from_this_card": false,
  "actor_retention_allowed": false,
  "promotion_allowed": false,
  "promotion_blocker": "BC slice fit is not a backtest; promotion still requires fixed-split and wf_2020/wf_2022/wf_2023_2024 replay with excess >= +30pp and MaxDD <= 15%.",
  "target_relative_return_pp": 30.0,
  "target_max_dd_pct": 15.0,
  "failed_checks": [],
  "next_mechanism": "run_walkforward_checkpoint_or_ppo_screen_with_bc_warmstart_against_soxlbacktest_reference_and_drawdown_drift_stop_rule

In [6]:
selection = json.loads((OUT_DIR / 'selection.json').read_text())
screen = pd.read_csv(OUT_DIR / 'bc_weight_screen.csv')
fold_metrics = pd.read_csv(OUT_DIR / 'fold_bc_metrics.csv')
slice_metrics = pd.read_csv(OUT_DIR / 'state_slice_bc_metrics.csv')
selected_config = selection['selected_config']
selected_row = screen[screen['config_name'].eq(selected_config)].iloc[0]
selected_folds = fold_metrics[fold_metrics['config_name'].eq(selected_config)]
selected_required = slice_metrics[
    slice_metrics['config_name'].eq(selected_config)
    & slice_metrics['slice'].isin(list(REQUIRED_SLICES))
]

required_lines = []
for name in REQUIRED_SLICES:
    rows = selected_required[selected_required['slice'].eq(name)]
    required_lines.append(
        f"- {name}: min accuracy {rows['accuracy'].min():.3f}, min samples {int(rows['samples'].min())}, gate {bool(selected_row[f'{name}_gate'])}"
    )

fold_lines = []
for row in selected_folds.itertuples(index=False):
    fold_lines.append(f"- {row.fold}: accuracy {row.accuracy:.3f}, samples {int(row.samples)}, gate {bool(row.fold_gate)}")

conclusion = f"""
## Conclusion

Decision: `{selection['decision']}`.

Selected BC config: `{selected_config}` with weighted fold accuracy `{selected_row['weighted_accuracy']:.3f}`, min fold accuracy `{selected_row['min_fold_accuracy']:.3f}`, and gate loss `{selected_row['gate_loss']:.4f}`.

Three-fold coverage: required folds `{', '.join(selection['required_folds'])}`; missing `{', '.join(selection['missing_required_folds']) if selection['missing_required_folds'] else 'none'}`; `wf_2020_coverage={selection['wf_2020_coverage']}`.

Fold fit:
""" + "\n".join(fold_lines) + "\n\nRequired sparse-state slice fit:\n" + "\n".join(required_lines) + f"""

Risk and promotion note: this is only a BC learnability screen. `promotion_allowed=false`, `actor_retention_allowed=false`, and `training_allowed_from_this_card=false`; a later checkpoint/PPO screen is allowed only when `bc_screen_gate=true`, and runtime promotion still requires fixed-split plus `wf_2020`, `wf_2022`, and `wf_2023_2024` replay against excess >= +30pp and MaxDD <= 15%.

Next action: `{selection['next_mechanism']}`.

Artifacts: `combined_target_contract.csv`, `bc_weight_screen.csv`, `state_slice_bc_metrics.csv`, `fold_bc_metrics.csv`, `bc_slice_summary.json`, `selection.json`, `summary.json`.
"""
(OUT_DIR / 'conclusion.md').write_text(conclusion.strip() + '\n')
display(Markdown(conclusion))


## Conclusion

Decision: `three_fold_bc_slice_ready_for_checkpoint_screen_no_promotion`.

Selected BC config: `release10_rebound4_downtrend2_steps600` with weighted fold accuracy `0.759`, min fold accuracy `0.727`, and gate loss `0.0040`.

Three-fold coverage: required folds `wf_2020_train, wf_2022_train, wf_2023_2024_train`; missing `none`; `wf_2020_coverage=True`.

Fold fit:
- wf_2020_train: accuracy 0.727, samples 2157, gate True
- wf_2022_train: accuracy 0.824, samples 505, gate True
- wf_2023_2024_train: accuracy 0.807, samples 755, gate True

Required sparse-state slice fit:
- release_gate_state: min accuracy 0.611, min samples 36, gate True
- is_downtrend_state: min accuracy 0.833, min samples 234, gate True
- drawdown_control_state: min accuracy 0.870, min samples 49, gate True

Risk and promotion note: this is only a BC learnability screen. `promotion_allowed=false`, `actor_retention_allowed=false`, and `training_allowed_from_this_card=false`; a later checkpoint/PPO screen is allowed only when `bc_screen_gate=true`, and runtime promotion still requires fixed-split plus `wf_2020`, `wf_2022`, and `wf_2023_2024` replay against excess >= +30pp and MaxDD <= 15%.

Next action: `run_walkforward_checkpoint_or_ppo_screen_with_bc_warmstart_against_soxlbacktest_reference_and_drawdown_drift_stop_rules`.

Artifacts: `combined_target_contract.csv`, `bc_weight_screen.csv`, `state_slice_bc_metrics.csv`, `fold_bc_metrics.csv`, `bc_slice_summary.json`, `selection.json`, `summary.json`.
